In [22]:
import OrcFxAPI
from pathlib import Path

owd_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2\Harlequin_convergence\OW_data\Harlequin_mesh1_owd.owd")
owr_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2\Harlequin_convergence\OW_results\Harlequin_mesh1_owr.owr")
xlsx_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2\Harlequin_convergence\XLSX_results\Harlequin_mesh1_results.xlsx")


In [23]:
water_depth = 200.0  # m
wave_periods = [1, 2, 3, 4,5,6,7,8,9,10,11,12,13, 14, 15,16, 17, 18, 19,20]   # s
wave_headings = [0.0]   # deg

mass = 65.61   # ton, alleen goed als jouw model ook ton verwacht
com_x = 0.0
com_y = 0.0
com_z = 6
# voorbeeld inertia matrix rond CoM
Ixx = 5061.21 
Iyy = 5061.21
Izz = 8346.92
Ixy = 0.0
Ixz = 0.0
Iyz = 0.0


# =========================
# HELPER FUNCTIONS
# =========================
def set_value(obj, name, value):
    """
    Zet een OrcaWave data item.
    De exacte data-itemnamen moet je uit OrcaWave halen met F7.
    """
    try:
        obj[name] = value
        print(f"Set: {name} = {value}")
    except Exception as e:
        print(f"Kon data item niet zetten: {name}")
        print(f"  Waarde: {value}")
        print(f"  Fout: {e}")


def print_validation(diff):
    info = getattr(diff, "ValidationInformationText", "")
    warnings = getattr(diff, "ValidationWarningText", "")
    errors = getattr(diff, "ValidationErrorText", "")

    print("\n=== VALIDATION INFO ===")
    print(info if info else "(geen info)")

    print("\n=== VALIDATION WARNINGS ===")
    print(warnings if warnings else "(geen warnings)")

    print("\n=== VALIDATION ERRORS ===")
    print(errors if errors else "(geen errors)")

    return errors


# =========================
# MAIN
# =========================
diff = OrcFxAPI.Diffraction()
diff.LoadData(str(owd_file))

# -------------------------------------------------
# BELANGRIJK:
# De namen hieronder zijn PLACEHOLDERS / typische structuur.
# Vervang ze met de exacte OrcaWave data names via F7 in de GUI.
# -------------------------------------------------

# Environment
diff.SetData("WaterDepth", 0, 200.0)

# Wave periods
# Vaak moet je eerst een count zetten en daarna de tabel vullen
diff.SetData("NumberOfPeriodsOrFrequencies", 0, len(wave_periods))
for i, T in enumerate(wave_periods):
    diff.SetData(f"PeriodOrFrequency", i, T)

# Wave headings
diff.SetData("NumberOfWaveHeadings", 0, len(wave_headings))
for i, hdg in enumerate(wave_headings):
    diff.SetData(f"WaveHeading", i, hdg)


diff.SetData("BodyInertiaSpecifiedBy", 0, "Matrix (for a general body)")
diff.SetData("BodyInertiaTensorOriginType", 0, "Centre of mass")
# Body inertia / mass properties
# Let op: body index / naam kan anders zijn in jouw model
diff.SetData("BodyCentreOfMassX", 0, com_x)
diff.SetData("BodyCentreOfMassY", 0, com_y)
diff.SetData("BodyCentreOfMassZ", 0, com_z)

diff.SetData("BodyMass", 0, mass)



diff.SetData("BodyInertiaTensorRx", 0, Ixx)   # xx
diff.SetData("BodyInertiaTensorRy", 1, Iyy)   # yy
diff.SetData("BodyInertiaTensorRz", 2, Izz)   # zz


print("Rx:", diff.BodyInertiaTensorRx[0])
print("Ry:", diff.BodyInertiaTensorRy[1])
print("Rz:", diff.BodyInertiaTensorRz[2])


# Optioneel: output settings
# Ook hier weer: exacte namen via F7
# set_value(diff, "CalculateDisplacementRAOs", "Yes")
# set_value(diff, "CalculateLoadRAOs", "Yes")
# set_value(diff, "CalculateAddedMassAndDamping", "Yes")
# set_value(diff, "CalculateWaveDriftQTFs", "No")

# Validation
errors = print_validation(diff)
if errors:
    raise RuntimeError("Model heeft validation errors. Fix eerst de data-itemnamen of invoer.")



Rx: 5061.21
Ry: 5061.21
Rz: 8346.92

=== VALIDATION INFO ===
('Estimated peak memory required during calculation: 6,8 MiB per thread.',)

=== VALIDATION WARNINGS ===
('Calculation mesh: the following panels are constructed from non-planar data in the mesh file: 2-4 10-12 18 20 26-28 34-36 42-44 50 52 58-60 66-68 74-76 82 84 90-92 98-100 106-108 114 116 122-124 130-132 138 140 146-148 154-156 162-164 170 172 178-180 186-188 194-196 202 204 210-212 218-220 226-228 234 236 242-244 250-252 258-260 266-268 274-276 282-284 290-292 298-300 306-308 314-316 322-324 330-332 338-340 346-348 354-356 362-364 370-372 378-380', 'Calculation mesh: the following panels are large compared to the wavelength of the shortest wave: 1-384', 'Mesh of Body1: irregular frequency effects are expected. Consider adding interior surface panels or removing short waves. The first irregular frequency is estimated at a period of 3,29699s (estimated for a body of length 20,9872m, breadth 23,7589m and draught 3,02m).')



In [24]:
# Run calculation
print("\n=== START CALCULATION ===")
diff.Calculate()
print("=== CALCULATION DONE ===")

# Save outputs
diff.SaveResults(str(owr_file))
diff.SaveResultsSpreadsheet(str(xlsx_file))

print("\nBestanden opgeslagen:")
print(f"  Results: {owr_file}")
print(f"  Spreadsheet: {xlsx_file}")


=== START CALCULATION ===
=== CALCULATION DONE ===

Bestanden opgeslagen:
  Results: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2\Harlequin_convergence\OW_results\Harlequin_mesh1_owr.owr
  Spreadsheet: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2\Harlequin_convergence\XLSX_results\Harlequin_mesh1_results.xlsx
